# VAE-OOD — Hyperspectral Target Detection via Reconstruction Error

**Adapted from:** Rouzoumka et al., *Out-of-Distribution Radar Detection with Complex VAEs: Theory, Whitening, and ANMF Fusion*, IEEE TSP 2025.

### Key idea
A VAE is trained **exclusively on background (H₀) pixel spectra**.  
At test time, target pixels are out-of-distribution and incur a **larger reconstruction error** than background pixels.  
This error serves as the detection score: `s(x) = ||x − x̂||²`.

### What this notebook does
1. Mounts Google Drive and installs `hyspan`
2. Loads a real HSI dataset
3. Extracts background pixels (using GT mask or coarse CEM)
4. Optionally applies PCA (recommended: reduces D before whitening)
5. Optionally applies local spatial whitening (paper Sec. IV-B)
6. Trains `SpectralVAE` on background pixels
7. Detects targets using reconstruction error
8. Fuses with CEM via weighted log-p (paper Eq. 18)
9. Reports AUC and plots results
10. Saves everything to Drive

### Drive layout expected
```
MyDrive/
  hyspan_datasets/
    Sandiego.mat          # 'data' (H,W,D), 'map' (H,W)
    abu-airport-2.mat
    pavia-u.mat           # optional
```

## 0 · Runtime check

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU.')

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT  = '/content/drive/MyDrive'
DATASET_DIR = os.path.join(DRIVE_ROOT, 'hyspan_datasets')
OUTPUT_DIR  = os.path.join(DRIVE_ROOT, 'hyspan_outputs', 'VAE_OOD')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Datasets:', os.listdir(DATASET_DIR))

## 2 · Install hyspan

In [ ]:
!pip install -q git+https://github.com/michaelpiro/hyspan.git
import importlib, site
importlib.invalidate_caches()

## 3 · Imports

In [ ]:
import os, json
import numpy as np
import scipy.io as sio
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from hyspan.deep_models.VAE_OOD import (
    SpectralVAE,
    train_vae_ood,
    vae_ood_detect,
    cem_detect,
    fuse_vae_cem,
    local_whiten_image,
)
from hyspan.results import roc_auc

print('hyspan imported OK')

## 4 · Configuration

In [ ]:
# ── Which dataset to process ──────────────────────────────────────────────
DATASET_FILE = 'Sandiego.mat'   # change to 'abu-airport-2.mat' etc.

# ── GT alignment ──────────────────────────────────────────────────────────
# Classic detectors confirmed the GT is spatially correct for all datasets.
# Set True only if you visually confirm misalignment AND classic detectors
# also show degraded AUC with the original GT.
GT_TRANSPOSE = False

# ── Preprocessing ─────────────────────────────────────────────────────────
PCA_COMPONENTS  = 30     # reduce to this many bands before training
                         # (None = skip PCA — NOT recommended when D >> 30)
USE_WHITENING   = True   # apply local spatial whitening (paper Sec. IV-B)
WHITEN_KERNEL   = 7      # k×k spatial neighbourhood for covariance estimation
WHITEN_RIDGE    = 1e-2   # regularisation fraction ε_ridge

# ── Background extraction ─────────────────────────────────────────────────
COARSE_BG_FRAC  = 0.6    # fraction of pixels used as pseudo-background when GT unavailable

# ── VAE architecture ──────────────────────────────────────────────────────
VAE_CFG = dict(
    latent_dim  = 16,
    hidden_dims = [256, 128],
    dropout     = 0.1,
)

# ── Training ──────────────────────────────────────────────────────────────
TRAIN_CFG = dict(
    epochs       = 100,
    batch_size   = 2048,
    lr           = 1e-3,
    weight_decay = 1e-5,
    beta         = 0.5,   # KL weight; <1 sharpens reconstruction (better OOD scores)
    device       = device,
    print_every  = 20,
)

# ── Fusion ────────────────────────────────────────────────────────────────
FUSION_W = 0.5   # weight on CEM branch (1 - FUSION_W on VAE branch)

## 5 · Helper utilities

In [ ]:
def load_mat(path):
    """Load .mat file → (img np.float32, gt np.float32 or None)."""
    mat = sio.loadmat(path)
    for key in ['data', 'Data', 'img', 'image', 'HSI', 'hsi']:
        if key in mat:
            img = mat[key].astype(np.float32); break
    else:
        keys = [k for k in mat if not k.startswith('_')]
        img  = mat[keys[0]].astype(np.float32)
    gt = None
    for key in ['map', 'Map', 'gt', 'GT', 'groundtruth', 'label']:
        if key in mat:
            gt = mat[key].astype(np.float32); break
    if img.shape[0] < img.shape[-1]:   # (D, H, W) → (H, W, D)
        img = img.transpose(1, 2, 0)
    # per-band [0,1] normalisation
    mn, mx = img.min(axis=(0,1), keepdims=True), img.max(axis=(0,1), keepdims=True)
    img = (img - mn) / (mx - mn + 1e-8)
    print(f'  image: {img.shape},  gt: {gt.shape if gt is not None else None}')
    return img, gt


def to_tensor(arr):
    """numpy → torch float32, safe on NumPy 2.x."""
    return torch.tensor(arr.tolist(), dtype=torch.float32)


def ensure_hwD(t: torch.Tensor) -> torch.Tensor:
    """Guarantee a 3-D tensor is (H, W, D) — spectral axis last."""
    if t.ndim != 3:
        return t
    a, b, c = t.shape
    if a == b and a != c: return t                                 # (H, W, D) ✓
    if a == c and a != b: return t.permute(0, 2, 1).contiguous()  # (H, D, W) → (H, W, D)
    if b == c and b != a: return t.permute(1, 2, 0).contiguous()  # (D, H, W) → (H, W, D)
    # all three differ — put axis with value most unlike the other two last
    dims  = [a, b, c]
    diffs = [abs(dims[i]-dims[(i+1)%3])+abs(dims[i]-dims[(i+2)%3]) for i in range(3)]
    d_axis = int(torch.tensor(diffs).argmax().item())
    if d_axis == 0: return t.permute(1, 2, 0).contiguous()
    if d_axis == 1: return t.permute(0, 2, 1).contiguous()
    return t


class TorchPCA:
    """Pure-torch PCA — avoids sklearn/numpy ABI issues."""
    def __init__(self, n_components):
        self.k = n_components
        self._components = None
        self._mean       = None

    def fit(self, pixels: torch.Tensor):             # (N, D)
        self._mean = pixels.mean(dim=0)
        X = pixels - self._mean
        cov = X.T @ X / (X.shape[0] - 1)
        _, vecs = torch.linalg.eigh(cov)
        self._components = vecs[:, -self.k:].flip(-1)  # (D, k)
        return self

    def transform(self, image: torch.Tensor) -> torch.Tensor:   # (H, W, D)
        H, W, D = image.shape
        x = image.reshape(-1, D) - self._mean
        return (x @ self._components).reshape(H, W, self.k)

    def transform_1d(self, v: torch.Tensor) -> torch.Tensor:    # (D,)
        return (v - self._mean) @ self._components               # (k,)


def compute_auc(scores: torch.Tensor, gt: torch.Tensor) -> float:
    s = scores.reshape(-1).numpy()
    g = gt.reshape(-1).numpy()
    if g.sum() == 0 or g.sum() == g.size:
        return float('nan')
    return roc_auc_score(g, s)


def plot_detections(image, gt, scores_dict, title, save_path=None):
    """Overlay multiple detection maps for comparison."""
    n = len(scores_dict)
    fig, axes = plt.subplots(1, n + 2, figsize=(4*(n+2), 4))
    fig.suptitle(title, fontsize=13)

    # False-colour RGB
    D = image.shape[-1]
    b1, b2, b3 = int(D*0.7), int(D*0.4), int(D*0.1)
    rgb = image[:, :, [b1, b2, b3]].numpy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    axes[0].imshow(rgb); axes[0].set_title('Image (false colour)'); axes[0].axis('off')

    # Ground truth
    axes[1].imshow(gt.numpy(), cmap='gray'); axes[1].set_title('Ground truth'); axes[1].axis('off')

    for ax, (name, scores) in zip(axes[2:], scores_dict.items()):
        ax.imshow(scores.numpy(), cmap='hot')
        ax.set_title(name); ax.axis('off')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()


print('Utilities ready.')

## 6 · Load dataset

In [ ]:
dataset_name = DATASET_FILE.replace('.mat', '')
mat_path     = os.path.join(DATASET_DIR, DATASET_FILE)
out_dir      = os.path.join(OUTPUT_DIR, dataset_name)
os.makedirs(out_dir, exist_ok=True)

print(f'Loading: {dataset_name}')
img_np, gt_np = load_mat(mat_path)

# Fix GT orientation if needed (some .mat files store GT with H/W swapped)
if GT_TRANSPOSE and gt_np is not None:
    gt_np = gt_np.T
    print(f'  GT transposed → {gt_np.shape}')

image = ensure_hwD(to_tensor(img_np))
gt    = to_tensor(gt_np) if gt_np is not None else None
H, W, D = image.shape
print(f'Image: {image.shape}   targets: {gt.sum():.0f}' if gt is not None else f'Image: {image.shape}  (no GT)')

# Quick alignment check: show image and GT side by side
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
fig.suptitle(f'{dataset_name} — alignment check (planes should overlap)', fontsize=11)
b1, b2, b3 = int(D*0.7), int(D*0.4), int(D*0.1)
rgb = image[:, :, [b1, b2, b3]].numpy()
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
axes[0].imshow(rgb); axes[0].set_title('False colour'); axes[0].axis('off')
axes[1].imshow(gt.numpy() if gt is not None else np.zeros((H, W)), cmap='hot')
axes[1].set_title('GT mask'); axes[1].axis('off')
plt.tight_layout(); plt.show()
print('If planes are misaligned, toggle GT_TRANSPOSE in cell 4.')

In [ ]:
"""
GT alignment verification — run this after loading, before any training.

Three independent checks:
  1. Visual overlay  — GT contour on false-colour image
  2. Spectral profiles — mean ± std of target vs background pixels
  3. CEM self-consistency — prior from gt==1, AUC should be >> 0.5 if GT correct

If CEM AUC < 0.6, the GT is likely transposed or otherwise wrong.
Toggle GT_TRANSPOSE in cell 4 and re-run from cell 6.
"""
if gt is None:
    print('No GT available — skipping alignment check.')
else:
    pixels = image.reshape(-1, D)
    gt_flat_v = gt.reshape(-1)
    tgt_px = pixels[gt_flat_v == 1]
    bg_px  = pixels[gt_flat_v == 0]

    # ── 1. Visual overlay ─────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(f'{dataset_name} — GT alignment check', fontsize=12)

    b1, b2, b3 = int(D*0.7), int(D*0.4), int(D*0.1)
    rgb = image[:, :, [b1, b2, b3]].numpy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)

    # left: false colour with GT contour overlaid in red
    axes[0].imshow(rgb)
    axes[0].contour(gt.numpy(), levels=[0.5], colors='red', linewidths=1.5)
    axes[0].set_title('False colour + GT contour (red)')
    axes[0].axis('off')

    # right: GT mask as hot map
    axes[1].imshow(gt.numpy(), cmap='hot')
    axes[1].set_title('GT mask')
    axes[1].axis('off')

    plt.tight_layout(); plt.show()

    # ── 2. Spectral profiles ──────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 3))
    bands = range(D)
    t_mean, t_std = tgt_px.mean(0).numpy(), tgt_px.std(0).numpy()
    b_mean, b_std = bg_px.mean(0).numpy(),  bg_px.std(0).numpy()
    ax.plot(bands, t_mean, 'r',   label=f'Target mean (n={tgt_px.shape[0]})')
    ax.fill_between(bands, t_mean-t_std, t_mean+t_std, alpha=0.2, color='red')
    ax.plot(bands, b_mean, 'b--', label=f'Background mean (n={bg_px.shape[0]})')
    ax.fill_between(bands, b_mean-b_std, b_mean+b_std, alpha=0.1, color='blue')
    ax.set_xlabel('Band'); ax.set_ylabel('Reflectance')
    ax.set_title('Spectral profiles — should look DIFFERENT if GT is correct')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    spectral_sep = float(((t_mean - b_mean)**2).mean()**0.5)
    print(f'Spectral distance (RMS of mean difference): {spectral_sep:.4f}')
    print(f'  > 0.05 suggests separable spectra (GT likely correct)')

    # ── 3. CEM self-consistency ───────────────────────────────────────────
    prior_v = tgt_px.mean(dim=0)
    cem_scores_v = cem_detect(image, prior_v, normalise=True)
    auc_cem_v    = compute_auc(cem_scores_v, gt)
    print(f'\nCEM self-consistency AUC: {auc_cem_v:.4f}')

    # ── Verdict ───────────────────────────────────────────────────────────
    print()
    if auc_cem_v > 0.75:
        verdict = '✓ GT appears CORRECTLY aligned'
    elif auc_cem_v > 0.55:
        verdict = '? GT alignment UNCERTAIN — inspect the overlay carefully'
    else:
        verdict = '✗ GT appears MISALIGNED (transposed?) — try GT_TRANSPOSE = True in cell 4'
    print('─' * 55)
    print(f'  {verdict}')
    print('─' * 55)

## 7 · PCA (optional but recommended)

In [ ]:
pca    = None
n_bands = D

# ── Step 1: per-band [0,1] normalisation is already done inside load_mat ──
# ── Step 2: PCA ───────────────────────────────────────────────────────────
if PCA_COMPONENTS is not None and D > PCA_COMPONENTS:
    print(f'PCA: {D} → {PCA_COMPONENTS} bands …')
    pca = TorchPCA(PCA_COMPONENTS)
    pca.fit(image.reshape(-1, D))
    image_pca = pca.transform(image)       # (H, W, k)  — values NOT in [0,1]
    n_bands   = PCA_COMPONENTS
    print(f'  PCA done: {image_pca.shape}  '
          f'value range [{image_pca.min():.2f}, {image_pca.max():.2f}]')
else:
    image_pca = image
    print(f'Skipping PCA — using all {D} bands.')

# ── Step 3: per-PC [0,1] normalisation AFTER PCA ─────────────────────────
# PCA projections are unbounded (approximately zero-mean, variance = eigenvalue).
# The VAE decoder uses Sigmoid (outputs [0,1]), so inputs must be in [0,1].
# We normalise each PC independently using the image-wide min/max.
# Important: this normalisation is computed once here and the same stats
# must be reused if you later transform new data (e.g., a real-eval image).
_px = image_pca.reshape(-1, n_bands)
pca_norm_min = _px.min(dim=0).values      # (n_bands,)  — save for reuse
pca_norm_max = _px.max(dim=0).values      # (n_bands,)
image_pca = ((_px - pca_norm_min) / (pca_norm_max - pca_norm_min + 1e-8)).reshape(image_pca.shape)
print(f'  After per-PC normalisation: range [{image_pca.min():.3f}, {image_pca.max():.3f}]  ✓')

## 8 · Local whitening (optional)

Analogous to the per-range-gate covariance whitening in the paper (Sec. IV-B, Alg. 1).
For each pixel, its k×k spatial neighbourhood is used to estimate a local covariance,
which is then used to whiten the pixel spectrum.

> **Note:** Works best after PCA (needs kernel² > n_bands for a full-rank SCM).

## 8b · Whitening comparison: None vs SCM vs GMM\n\nTrains one `SpectralVAE` per whitening variant and compares AUC.\n\n| Variant | How covariance is estimated |\n|---|---|\n| **None** | No whitening — raw PCA-reduced spectra |\n| **SCM** | k×k patch sample covariance (paper Sec. IV-B) |\n| **GMM** | Global GMM → majority-vote component → per-component Cholesky |\n\nThe GMM whitening naturally handles multi-modal backgrounds (runways, grass, water) because each cluster's covariance is estimated from all similar pixels — not just a tiny spatial patch."

In [ ]:
from hyspan.deep_models.VAE_OOD import gmm_whiten_image

# ── Config for comparison ──────────────────────────────────────────────────
COMPARE_EPOCHS = 60    # fewer epochs for speed (increase for final results)
COMPARE_KERNEL = 7
COMPARE_N_COMP = 10    # GMM components

# ── Helper: normalize bands to [0,1] after whitening ─────────────────────
# Whitened pixel values are unbounded (they are approximately N(0,I) for
# background). The VAE decoder uses Sigmoid which outputs [0,1], so we
# rescale each band of the whitened image to [0,1] before training.
# This does NOT undo the whitening — the relative structure is preserved.
def normalize_bands_01(img: torch.Tensor) -> torch.Tensor:
    """Per-band min-max normalize a (H, W, D) image to [0, 1]."""
    H, W, D = img.shape
    px  = img.reshape(-1, D)
    mn  = px.min(dim=0).values
    mx  = px.max(dim=0).values
    return ((px - mn) / (mx - mn + 1e-8)).reshape(H, W, D)

# ── Derive background pixels from image_pca + gt ─────────────────────────
_px_pca      = image_pca.reshape(-1, n_bands)
_gt_flat_cmp = gt.reshape(-1) if gt is not None else None
_bg_px_cmp   = _px_pca[_gt_flat_cmp == 0] if _gt_flat_cmp is not None else _px_pca
_gt_mask_cmp = _gt_flat_cmp

# ── Build the three image variants ────────────────────────────────────────
print('Preparing whitening variants …')

# 1. No whitening
image_none = image_pca

# 2. SCM whitening  (normalize afterwards so VAE Sigmoid decoder works)
print('SCM whitening …')
image_scm = normalize_bands_01(local_whiten_image(
    image_pca,
    kernel_size = COMPARE_KERNEL,
    ridge_frac  = WHITEN_RIDGE,
    device      = device,
))
print(f'  SCM whitened + normalized: {image_scm.shape}  range=[{image_scm.min():.2f}, {image_scm.max():.2f}]')

# 3. GMM whitening  (normalize afterwards for the same reason)
print('GMM whitening …')
image_gmm_raw, fitted_gmm = gmm_whiten_image(
    image_pca,
    n_components = COMPARE_N_COMP,
    kernel_size  = COMPARE_KERNEL,
    fit_pixels   = _bg_px_cmp,
    device       = device,
)
image_gmm = normalize_bands_01(image_gmm_raw)
print(f'  GMM whitened + normalized: {image_gmm.shape}  range=[{image_gmm.min():.2f}, {image_gmm.max():.2f}]')

variants = {
    'None (PCA only)': image_none,
    'SCM whitening':   image_scm,
    'GMM whitening':   image_gmm,
}
print('All variants ready.')

# ── Train VAE on each variant and evaluate ────────────────────────────────
def get_bg_pixels(img, gt_mask):
    px = img.reshape(-1, img.shape[-1])
    return px[gt_mask == 0] if gt_mask is not None else px

comparison_results = {}

for name, img_variant in variants.items():
    print(f'\n─── {name} ───')
    D_v     = img_variant.shape[-1]
    bg_v    = get_bg_pixels(img_variant, _gt_mask_cmp)
    prior_v = img_variant.reshape(-1, D_v)[_gt_mask_cmp == 1].mean(dim=0) \
              if _gt_mask_cmp is not None and (_gt_mask_cmp == 1).any() \
              else img_variant.reshape(-1, D_v).mean(dim=0)

    # Train VAE
    m_v = SpectralVAE(n_bands=D_v, **VAE_CFG).to(device)
    history_v = train_vae_ood(
        m_v, bg_v,
        epochs=COMPARE_EPOCHS, batch_size=2048, lr=1e-3,
        beta=TRAIN_CFG['beta'], device=device, print_every=COMPARE_EPOCHS,
    )

    # VAE scores
    s_vae_raw = vae_ood_detect(m_v, img_variant, normalise=False, device=device).cpu()
    lo, hi    = s_vae_raw.min(), s_vae_raw.max()
    s_vae     = (s_vae_raw - lo) / (hi - lo + 1e-12)

    # CEM scores
    s_cem_raw = cem_detect(img_variant, prior_v, normalise=False).cpu()
    lo, hi    = s_cem_raw.min(), s_cem_raw.max()
    s_cem     = (s_cem_raw - lo) / (hi - lo + 1e-12)

    # Fusion
    bg_vae_s = s_vae_raw.reshape(-1)[_gt_mask_cmp == 0] if _gt_mask_cmp is not None else s_vae_raw.reshape(-1)
    bg_cem_s = s_cem_raw.reshape(-1)[_gt_mask_cmp == 0] if _gt_mask_cmp is not None else s_cem_raw.reshape(-1)
    s_fused  = fuse_vae_cem(s_vae_raw, s_cem_raw, bg_vae_s, bg_cem_s, w=FUSION_W, normalise=True).cpu()

    gt_ref  = gt if gt is not None else torch.zeros(H, W)
    auc_vae   = compute_auc(s_vae,   gt_ref)
    auc_cem   = compute_auc(s_cem,   gt_ref)
    auc_fused = compute_auc(s_fused, gt_ref)

    comparison_results[name] = dict(
        auc_vae=auc_vae, auc_cem=auc_cem, auc_fused=auc_fused,
        s_vae=s_vae, s_cem=s_cem, s_fused=s_fused, history=history_v,
    )
    print(f'  VAE-OOD: {auc_vae:.4f}  |  CEM: {auc_cem:.4f}  |  Fused: {auc_fused:.4f}')

# ── Summary table ──────────────────────────────────────────────────────────
print('\n' + '='*62)
print(f'{"Whitening":20s}  {"VAE-OOD":>10}  {"CEM":>8}  {"Fused":>8}')
print('='*62)
for name, res in comparison_results.items():
    print(f'{name:20s}  {res["auc_vae"]:>10.4f}  {res["auc_cem"]:>8.4f}  {res["auc_fused"]:>8.4f}')
print('='*62)

In [ ]:
# ── Visual comparison: VAE detection maps for each whitening variant ───────
n_variants = len(comparison_results)
fig, axes = plt.subplots(3, n_variants + 1, figsize=(4*(n_variants+1), 10))
fig.suptitle(f'Whitening comparison — {dataset_name}', fontsize=13)

row_labels = ['VAE-OOD', 'CEM', 'Fused']
score_keys = ['s_vae', 's_cem', 's_fused']
auc_keys   = ['auc_vae', 'auc_cem', 'auc_fused']

# Column 0: GT and image
D_orig = image.shape[-1]
b1, b2, b3 = int(D_orig*0.7), int(D_orig*0.4), int(D_orig*0.1)
rgb = image[:, :, [b1, b2, b3]].numpy()
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
axes[0, 0].imshow(rgb); axes[0, 0].set_title('False colour'); axes[0, 0].axis('off')
axes[1, 0].imshow(gt.numpy() if gt is not None else torch.zeros(H, W).numpy(), cmap='gray')
axes[1, 0].set_title('Ground truth'); axes[1, 0].axis('off')
axes[2, 0].axis('off')

for col, (name, res) in enumerate(comparison_results.items(), start=1):
    for row, (row_label, sk, ak) in enumerate(zip(row_labels, score_keys, auc_keys)):
        ax  = axes[row, col]
        auc = res[ak]
        ax.imshow(res[sk].numpy(), cmap='hot')
        ax.set_title(f'{row_label}\n{name}\nAUC={auc:.3f}', fontsize=8)
        ax.axis('off')

plt.tight_layout()
save_path = os.path.join(out_dir, 'whitening_comparison.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Saved → {save_path}')
plt.show()

# ── Bar chart: AUC by whitening and detector ──────────────────────────────
names    = list(comparison_results.keys())
auc_vae_vals   = [comparison_results[n]['auc_vae']   for n in names]
auc_cem_vals   = [comparison_results[n]['auc_cem']   for n in names]
auc_fused_vals = [comparison_results[n]['auc_fused'] for n in names]

x     = range(len(names))
width = 0.25
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([i - width for i in x], auc_vae_vals,   width, label='VAE-OOD', color='steelblue')
ax.bar([i         for i in x], auc_cem_vals,   width, label='CEM',     color='coral')
ax.bar([i + width for i in x], auc_fused_vals, width, label='Fused',   color='seagreen')
ax.set_xticks(list(x)); ax.set_xticklabels(names, rotation=10)
ax.set_ylabel('AUC'); ax.set_ylim(0, 1)
ax.set_title(f'AUC by whitening strategy — {dataset_name}')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'whitening_auc_bars.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
if USE_WHITENING:
    print(f'Local whitening: kernel={WHITEN_KERNEL}×{WHITEN_KERNEL}, ridge={WHITEN_RIDGE} …')
    if n_bands > WHITEN_KERNEL ** 2:
        print(f'  WARNING: n_bands ({n_bands}) > kernel² ({WHITEN_KERNEL**2}). '
              'SCM will be rank-deficient — ridge regularisation compensates, '
              'but increase kernel or reduce n_bands via PCA for best results.')
    image_proc = local_whiten_image(
        image_pca,
        kernel_size  = WHITEN_KERNEL,
        ridge_frac   = WHITEN_RIDGE,
        device       = device,
    )
    print(f'  Whitened image: {image_proc.shape}')
else:
    image_proc = image_pca
    print('Skipping whitening.')

n_bands_proc = image_proc.shape[-1]

## 9 · Extract background pixels (H₀ training set)

In [ ]:
pixels_all = image_proc.reshape(-1, n_bands_proc)   # (H*W, D')

if gt is not None:
    gt_flat    = gt.reshape(-1)
    bg_mask    = (gt_flat == 0)
    bg_pixels  = pixels_all[bg_mask]
    tgt_pixels = pixels_all[~bg_mask]
    print(f'GT-based background: {bg_pixels.shape[0]:,} px  |  targets: {tgt_pixels.shape[0]:,} px')
else:
    # Coarse CEM pass — no GT available
    print('No GT — using coarse CEM to select background pixels …')
    t   = image.reshape(-1, D).mean(dim=0)   # crude prior: mean spectrum (placeholder)
    R   = pixels_all.T @ pixels_all / pixels_all.shape[0]
    R  += 1e-6 * torch.eye(n_bands_proc)
    c   = torch.linalg.solve(R, t[:n_bands_proc].unsqueeze(-1)).squeeze(-1)
    c   = c / (t[:n_bands_proc] @ c + 1e-12)
    cem_coarse = (pixels_all @ c)
    n_bg       = int(COARSE_BG_FRAC * pixels_all.shape[0])
    bg_idx     = torch.argsort(cem_coarse)[:n_bg]
    bg_pixels  = pixels_all[bg_idx]
    gt_flat    = torch.zeros(H * W)
    print(f'Pseudo-background (bottom {COARSE_BG_FRAC:.0%} CEM): {bg_pixels.shape[0]:,} px')

# Prior spectrum from known target pixels (for CEM and fusion)
if gt is not None and (gt_flat == 1).any():
    # Use the original (pre-processed) pixel space for the prior
    prior = tgt_pixels.mean(dim=0)             # (D',)
    print(f'Prior from {tgt_pixels.shape[0]} GT target pixels')
else:
    # No GT — use mean spectrum of pixels with highest CEM scores
    n_tgt = max(10, int(0.005 * pixels_all.shape[0]))
    prior = pixels_all[torch.argsort(cem_coarse)[-n_tgt:]].mean(dim=0)
    print(f'Prior from top-{n_tgt} CEM pixels (no GT available)')

## 10 · Train SpectralVAE on background pixels

In [ ]:
model = SpectralVAE(n_bands=n_bands_proc, **VAE_CFG)
n_p   = sum(p.numel() for p in model.parameters())
print(f'SpectralVAE  |  bands={n_bands_proc}  latent={VAE_CFG["latent_dim"]}  params={n_p:,}')

ckpt_path = os.path.join(out_dir, 'vae_best.pt')
print(f'Training on {bg_pixels.shape[0]:,} background pixels …')
history = train_vae_ood(
    model, bg_pixels, checkpoint_path=ckpt_path, **TRAIN_CFG
)
print(f'Training done. Best checkpoint → {ckpt_path}')

# Save history
with open(os.path.join(out_dir, 'history.json'), 'w') as f:
    json.dump(history, f, indent=2)

# Plot loss curve
epochs   = [m['epoch'] for m in history]
total_l  = [m['loss']  for m in history]
rec_l    = [m['l_rec'] for m in history]
kl_l     = [m['l_kl']  for m in history]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, total_l, label='Total ELBO')
ax.plot(epochs, rec_l,   label='Reconstruction', linestyle='--')
ax.plot(epochs, kl_l,    label='KL', linestyle=':')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title(f'Training — {dataset_name}'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'loss_curve.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11 · Detection: VAE reconstruction error

In [ ]:
print('Computing VAE reconstruction error map …')
scores_vae_raw = vae_ood_detect(
    model, image_proc, batch_size=4096, normalise=False, device=device
).cpu()

# Normalised version for display
lo, hi = scores_vae_raw.min(), scores_vae_raw.max()
scores_vae = (scores_vae_raw - lo) / (hi - lo + 1e-12)

auc_vae = compute_auc(scores_vae, gt if gt is not None else torch.zeros(H, W))
print(f'VAE-OOD AUC: {auc_vae:.4f}')

## 12 · Detection: CEM baseline

In [ ]:
print('Computing CEM scores …')
scores_cem_raw = cem_detect(image_proc, prior, normalise=False).cpu()

lo, hi = scores_cem_raw.min(), scores_cem_raw.max()
scores_cem = (scores_cem_raw - lo) / (hi - lo + 1e-12)

auc_cem = compute_auc(scores_cem, gt if gt is not None else torch.zeros(H, W))
print(f'CEM AUC: {auc_cem:.4f}')

## 13 · Fusion: weighted log-p (paper Eq. 18)

```
S* = -(w · log p_CEM + (1-w) · log p_VAE)
```
Both p-values are right-tail empirical CDFs calibrated on background pixels (H₀).

In [ ]:
# Background scores for H0 calibration
bg_vae_scores = scores_vae_raw.reshape(-1)[gt_flat == 0]
bg_cem_scores = scores_cem_raw.reshape(-1)[gt_flat == 0]

print(f'Fusing VAE + CEM  (w_CEM={FUSION_W}, w_VAE={1-FUSION_W}) …')
scores_fused = fuse_vae_cem(
    scores_vae_raw, scores_cem_raw,
    bg_vae_scores, bg_cem_scores,
    w=FUSION_W, normalise=True,
).cpu()

auc_fused = compute_auc(scores_fused, gt if gt is not None else torch.zeros(H, W))
print(f'Fused AUC: {auc_fused:.4f}')

# Summary table
print()
print('─' * 35)
print(f'  CEM              AUC = {auc_cem:.4f}')
print(f'  VAE-OOD          AUC = {auc_vae:.4f}')
print(f'  VAE+CEM (fused)  AUC = {auc_fused:.4f}')
print('─' * 35)

## 14 · Plot and save results

In [ ]:
scores_dict = {
    f'CEM (AUC={auc_cem:.3f})':          scores_cem,
    f'VAE-OOD (AUC={auc_vae:.3f})':      scores_vae,
    f'Fused (AUC={auc_fused:.3f})':      scores_fused,
}

plot_detections(
    image,                           # original image (for false-colour display)
    gt if gt is not None else torch.zeros(H, W),
    scores_dict,
    title=f'VAE-OOD — {dataset_name}',
    save_path=os.path.join(out_dir, 'detection_maps.png'),
)

# Save score tensors
torch.save({
    'scores_vae':   scores_vae,
    'scores_cem':   scores_cem,
    'scores_fused': scores_fused,
    'gt':           gt,
    'auc_vae':      auc_vae,
    'auc_cem':      auc_cem,
    'auc_fused':    auc_fused,
}, os.path.join(out_dir, 'results.pt'))
print(f'All results saved to {out_dir}')

## 15 · ROC curves

In [ ]:
if gt is not None and gt.sum() > 0:
    from sklearn.metrics import roc_curve

    gt_flat_np = gt.reshape(-1).numpy()

    fig, ax = plt.subplots(figsize=(6, 6))
    for name, scores in [
        ('CEM',          scores_cem),
        ('VAE-OOD',      scores_vae),
        ('VAE+CEM fused', scores_fused),
    ]:
        fpr, tpr, _ = roc_curve(gt_flat_np, scores.reshape(-1).numpy())
        auc_v       = compute_auc(scores, gt)
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc_v:.3f})')

    ax.plot([0,1],[0,1],'k--', alpha=0.4)
    ax.set_xlabel('False Alarm Rate')
    ax.set_ylabel('Detection Probability')
    ax.set_title(f'ROC — {dataset_name}')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'roc_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No GT available — skipping ROC curves.')

## 16 · Score distribution: background vs target

Shows how well the VAE reconstruction score separates background from target.

In [ ]:
if gt is not None and gt.sum() > 0:
    gt_flat_bool = (gt_flat == 1)
    s_flat       = scores_vae.reshape(-1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'Score distributions — {dataset_name}', fontsize=13)

    for ax, (name, s) in zip(axes, [
        ('CEM',          scores_cem.reshape(-1)),
        ('VAE-OOD',      scores_vae.reshape(-1)),
        ('VAE+CEM fused', scores_fused.reshape(-1)),
    ]):
        bg_s  = s[~gt_flat_bool].numpy()
        tgt_s = s[gt_flat_bool].numpy()
        bins  = np.linspace(s.min().item(), s.max().item(), 60)
        ax.hist(bg_s,  bins=bins, alpha=0.5, color='blue',  density=True, label='Background')
        ax.hist(tgt_s, bins=bins, alpha=0.7, color='red',   density=True, label='Target')
        ax.set_title(name); ax.set_xlabel('Score'); ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'score_distributions.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No GT — skipping distribution plot.')

## 17 · Ablation: vary β and fusion weight

Quick sweep over key hyperparameters to understand sensitivity.

In [ ]:
# ── β sweep (KL weight) — trains multiple small VAEs ─────────────────────
if gt is not None and gt.sum() > 0:
    beta_values = [0.1, 0.5, 1.0, 2.0]
    auc_beta    = []

    for beta_v in beta_values:
        print(f'  β = {beta_v} …', end=' ', flush=True)
        m_tmp = SpectralVAE(n_bands=n_bands_proc, **VAE_CFG).to(device)
        _ = train_vae_ood(
            m_tmp, bg_pixels,
            epochs=50, batch_size=2048, lr=1e-3, beta=beta_v,
            device=device, print_every=999,
        )
        s_tmp = vae_ood_detect(m_tmp, image_proc, normalise=True, device=device).cpu()
        auc_tmp = compute_auc(s_tmp, gt)
        auc_beta.append(auc_tmp)
        print(f'AUC={auc_tmp:.4f}')

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(beta_values, auc_beta, 'o-')
    ax.set_xlabel('β (KL weight)'); ax.set_ylabel('AUC')
    ax.set_title(f'β sensitivity — {dataset_name}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'beta_sweep.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Fusion weight sweep ────────────────────────────────────────────────
    w_values  = [0.0, 0.25, 0.5, 0.75, 1.0]
    auc_fuse  = []
    for w_v in w_values:
        s_f = fuse_vae_cem(
            scores_vae_raw, scores_cem_raw,
            bg_vae_scores, bg_cem_scores,
            w=w_v, normalise=True,
        ).cpu()
        auc_fuse.append(compute_auc(s_f, gt))
        print(f'  w_CEM={w_v:.2f}  AUC={auc_fuse[-1]:.4f}')

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(w_values, auc_fuse, 's-', color='green')
    ax.axhline(auc_vae, linestyle='--', color='blue', alpha=0.7, label=f'VAE only ({auc_vae:.3f})')
    ax.axhline(auc_cem, linestyle='--', color='red',  alpha=0.7, label=f'CEM only ({auc_cem:.3f})')
    ax.set_xlabel('w_CEM (fusion weight)'); ax.set_ylabel('AUC')
    ax.set_title(f'Fusion weight sensitivity — {dataset_name}')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'fusion_weight_sweep.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No GT — skipping ablation.')

## 18 · Reconstruct a few example spectra

Visual sanity check: background pixels should reconstruct well; target pixels should not.

## 19 · Evaluate on synthetic images (from THANTD session)

The THANTD notebook synthesised `(500×250)` images from each real dataset using `SynthesisEngine`
and saved them to `MyDrive/hyspan_outputs/<dataset_name>/synthetic_full.pt`.

Here we load those images and run the **same VAE trained on the real background pixels** against
them. This is the reverse of the THANTD transfer experiment: instead of training on synthetic
and testing on real, we train on real and test on synthetic.

Pipeline applied to each synthetic image before detection:
1. Apply the same `pca` fitted on the real image  
2. Apply the same `pca_norm_min / pca_norm_max` scaling  
3. Apply whitening (same method as the main pipeline: SCM or none)  

A high AUC here means the synthetic targets are spectrally distinguishable from synthetic
background **as seen through the real-image VAE** — i.e. the synthesis preserved the OOD
structure of targets.

In [ ]:
# ── Path to synthetic images saved by the THANTD notebook ─────────────────
# THANTD used OUTPUT_DIR = MyDrive/hyspan_outputs/<dataset_name>/
THANTD_OUTPUT_DIR = os.path.join(DRIVE_ROOT, 'hyspan_outputs')

def preprocess_synthetic(synth_image: torch.Tensor) -> torch.Tensor:
    """
    Apply to a synthetic (H, W, D_orig) image the exact same preprocessing
    pipeline that was used on the real image before VAE training:
      1. PCA (if fitted)
      2. Per-PC [0,1] normalisation (using the same min/max as the real image)
      3. Local whitening (if USE_WHITENING)
    """
    H, W, D_s = synth_image.shape
    assert D_s == D, (
        f'Synthetic image has {D_s} bands but real image has {D} bands — '
        'they must come from the same dataset.'
    )

    # Step 1: PCA
    if pca is not None:
        img_p = pca.transform(synth_image)          # (H, W, n_bands)
    else:
        img_p = synth_image.clone()

    # Step 2: per-PC normalisation with REAL-image stats
    # Using the real pca_norm_min/max ensures the synthetic data lives in
    # the same [0,1] space as the training data the VAE saw.
    p = img_p.reshape(-1, n_bands)
    img_p = ((p - pca_norm_min) / (pca_norm_max - pca_norm_min + 1e-8)).reshape(H, W, n_bands)

    # Step 3: whitening (recomputed on synthetic image — whitening is local
    # and does not use parameters from the real image)
    if USE_WHITENING:
        img_p = local_whiten_image(
            img_p, kernel_size=WHITEN_KERNEL, ridge_frac=WHITEN_RIDGE, device=device,
        )
        # Normalize back to [0,1] (whitening makes values unbounded)
        p2 = img_p.reshape(-1, img_p.shape[-1])
        mn, mx = p2.min(dim=0).values, p2.max(dim=0).values
        img_p = ((p2 - mn) / (mx - mn + 1e-8)).reshape(img_p.shape)

    return img_p


# ── Discover available synthetic files for the current dataset ─────────────
synth_dir_thantd = os.path.join(THANTD_OUTPUT_DIR, dataset_name)

# Try full image first, fall back to test split
synth_candidates = [
    ('Full (500×250)',  os.path.join(synth_dir_thantd, 'synthetic_full.pt')),
    ('Test split (150×250)', os.path.join(synth_dir_thantd, 'test.pt')),
]

synth_results = {}

for split_name, synth_path in synth_candidates:
    if not os.path.exists(synth_path):
        print(f'Not found: {synth_path} — skipping.')
        continue

    print(f'\nLoading synthetic image ({split_name}): {synth_path}')
    synth_data  = torch.load(synth_path, map_location='cpu')
    synth_image = synth_data['image']    # (H_s, W_s, D_orig)
    synth_gt    = synth_data['gt']       # (H_s, W_s)
    H_s, W_s, D_s = synth_image.shape
    n_tgt_s = synth_gt.sum().int().item()
    print(f'  Shape: {synth_image.shape}   targets: {n_tgt_s}')

    if D_s != D:
        print(f'  ✗ Band mismatch (synthetic={D_s}, real={D}) — skipping.')
        continue
    if n_tgt_s == 0:
        print(f'  ✗ No target pixels in this split — skipping.')
        continue

    # ── Preprocess ────────────────────────────────────────────────────────
    print('  Preprocessing (PCA + normalisation + whitening) …')
    synth_proc = preprocess_synthetic(synth_image)

    # ── VAE detection ─────────────────────────────────────────────────────
    s_vae_raw_s = vae_ood_detect(model, synth_proc, normalise=False, device=device).cpu()
    lo, hi      = s_vae_raw_s.min(), s_vae_raw_s.max()
    s_vae_s     = (s_vae_raw_s - lo) / (hi - lo + 1e-12)
    auc_vae_s   = compute_auc(s_vae_s, synth_gt)

    # ── CEM detection ─────────────────────────────────────────────────────
    # Prior: target mean in the PROCESSED synthetic space
    synth_px    = synth_proc.reshape(-1, synth_proc.shape[-1])
    synth_gt_f  = synth_gt.reshape(-1)
    prior_s     = synth_px[synth_gt_f == 1].mean(dim=0)

    s_cem_raw_s = cem_detect(synth_proc, prior_s, normalise=False).cpu()
    lo, hi      = s_cem_raw_s.min(), s_cem_raw_s.max()
    s_cem_s     = (s_cem_raw_s - lo) / (hi - lo + 1e-12)
    auc_cem_s   = compute_auc(s_cem_s, synth_gt)

    # ── Fusion ────────────────────────────────────────────────────────────
    bg_vae_s2 = s_vae_raw_s.reshape(-1)[synth_gt_f == 0]
    bg_cem_s2 = s_cem_raw_s.reshape(-1)[synth_gt_f == 0]
    s_fused_s = fuse_vae_cem(
        s_vae_raw_s, s_cem_raw_s, bg_vae_s2, bg_cem_s2,
        w=FUSION_W, normalise=True,
    ).cpu()
    auc_fused_s = compute_auc(s_fused_s, synth_gt)

    print(f'  VAE-OOD: {auc_vae_s:.4f}  |  CEM: {auc_cem_s:.4f}  |  Fused: {auc_fused_s:.4f}')
    synth_results[split_name] = dict(
        image=synth_image, gt=synth_gt, proc=synth_proc,
        s_vae=s_vae_s, s_cem=s_cem_s, s_fused=s_fused_s,
        auc_vae=auc_vae_s, auc_cem=auc_cem_s, auc_fused=auc_fused_s,
    )

if not synth_results:
    print('\nNo synthetic files found. Run the THANTD notebook first to generate them.')
else:
    # ── Plot ──────────────────────────────────────────────────────────────
    for split_name, sr in synth_results.items():
        fig, axes = plt.subplots(1, 5, figsize=(22, 4))
        fig.suptitle(
            f'VAE (trained on REAL {dataset_name}) → SYNTHETIC {split_name}', fontsize=12
        )

        # False colour of synthetic image
        rgb_s = sr['image'][:, :, [int(D*0.7), int(D*0.4), int(D*0.1)]].numpy()
        rgb_s = (rgb_s - rgb_s.min()) / (rgb_s.max() - rgb_s.min() + 1e-8)
        axes[0].imshow(rgb_s); axes[0].set_title('Synthetic image'); axes[0].axis('off')

        # GT
        axes[1].imshow(sr['gt'].numpy(), cmap='gray')
        axes[1].set_title('Synthetic GT'); axes[1].axis('off')

        # Detection maps
        for ax, (name, scores, auc_v) in zip(axes[2:], [
            ('VAE-OOD', sr['s_vae'],   sr['auc_vae']),
            ('CEM',     sr['s_cem'],   sr['auc_cem']),
            ('Fused',   sr['s_fused'], sr['auc_fused']),
        ]):
            ax.imshow(scores.numpy(), cmap='hot')
            ax.set_title(f'{name}\nAUC={auc_v:.4f}'); ax.axis('off')

        plt.tight_layout()
        save_name = split_name.replace(' ', '_').replace('(', '').replace(')', '').replace('×', 'x')
        plt.savefig(os.path.join(out_dir, f'synth_eval_{save_name}.png'), dpi=150, bbox_inches='tight')
        plt.show()

    # ── Summary ───────────────────────────────────────────────────────────
    print('\n' + '='*60)
    print(f'Synthetic evaluation — VAE trained on REAL {dataset_name}')
    print('='*60)
    print(f'{"Split":25s}  {"VAE-OOD":>8}  {"CEM":>8}  {"Fused":>8}')
    print('-'*60)
    for split_name, sr in synth_results.items():
        print(f'{split_name:25s}  {sr["auc_vae"]:>8.4f}  {sr["auc_cem"]:>8.4f}  {sr["auc_fused"]:>8.4f}')
    print('-'*60)
    print(f'{"Real image (reference)":25s}  {auc_vae:>8.4f}  {auc_cem:>8.4f}  {auc_fused:>8.4f}')
    print('='*60)
    print()
    print('Interpretation:')
    print('  If synthetic AUC >> real AUC: VAE finds synthetic targets easy (good synthesis).')
    print('  If synthetic AUC ≈ real AUC:  Synthesis faithfully reproduces OOD structure.')
    print('  If synthetic AUC << real AUC: Synthetic targets blend into the background —')
    print('                                synthesis gap is too large for this VAE.')

In [ ]:
model.eval()
model.to(device)

n_show = 4

with torch.no_grad():
    # Sample background pixels with lowest reconstruction error
    bg_scores_sorted = scores_vae.reshape(-1)[gt_flat == 0].sort()
    bg_px_sorted     = bg_pixels[bg_scores_sorted.indices[:n_show]].to(device)
    bg_hat, _, _     = model(bg_px_sorted)

    # Sample target pixels (if GT available)
    if gt is not None and gt.sum() > 0:
        tgt_px   = pixels_all[gt_flat == 1][:n_show].to(device)
        tgt_hat, _, _ = model(tgt_px)

fig, axes = plt.subplots(2, n_show, figsize=(4*n_show, 6))
fig.suptitle('Reconstruction quality: background (top) vs target (bottom)', fontsize=12)
bands = range(n_bands_proc)

for i in range(n_show):
    ax = axes[0, i]
    ax.plot(bands, bg_px_sorted[i].cpu().numpy(), label='Input', alpha=0.8)
    ax.plot(bands, bg_hat[i].cpu().numpy(),        label='Recon', linestyle='--', alpha=0.8)
    err = ((bg_px_sorted[i].cpu() - bg_hat[i].cpu())**2).sum().item()
    ax.set_title(f'BG pixel {i+1}  err={err:.4f}'); ax.set_xlabel('Band')
    if i == 0: ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    if gt is not None and gt.sum() > 0 and i < tgt_px.shape[0]:
        ax = axes[1, i]
        ax.plot(bands, tgt_px[i].cpu().numpy(), label='Input', alpha=0.8, color='red')
        ax.plot(bands, tgt_hat[i].cpu().numpy(), label='Recon', linestyle='--', alpha=0.8, color='orange')
        err = ((tgt_px[i].cpu() - tgt_hat[i].cpu())**2).sum().item()
        ax.set_title(f'Target pixel {i+1}  err={err:.4f}'); ax.set_xlabel('Band')
        if i == 0: ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    else:
        axes[1, i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'spectrum_reconstructions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Target pixels should have HIGHER error than background pixels.')